# Posist API Data Ingestion: Source to Landing

This notebook ingests bills data from the Posist API in JSON format. It uses configuration files for each outlet, enabling outlet-specific ingestion parameters. The process includes a watermark mechanism to track and manage incremental loads efficiently. Additionally, the notebook supports historical data loads by fetching and processing data for past dates as required.

**Key capabilities:**
- Config-driven ingestion — one config JSON per outlet
- Watermark-based incremental loads
- Paginated API response handling
- Historical data load support
- Writes raw JSON responses to ADLS Gen2 landing layer

## Import utilities

In [ ]:
# Import utility for API interaction and ADLS writes
%run /Workspace/Shared/posist/utilities/utility_api


In [ ]:
# Import utility for timestamp functions
%run /Workspace/Shared/posist/utilities/utility_timestamps


## Import necessary packages

In [ ]:
# Import packages and setup logging
import json
import logging
from pyspark.sql import Row
from datetime import datetime, timedelta

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("logger")

## Helper functions

In [ ]:
# Function for fetching bills
# This GET request retrieves bill information from a specific resource endpoint in the API
def get_bills(access_token, customer_key, start_timestamp, end_timestamp, page):
    """
    Fetches paginated bill data from the Posist API for a given time range.

    Args:
        access_token (str): Bearer token for API authentication.
        customer_key (str): Outlet-specific customer key.
        start_timestamp (int): Start of time range in Unix ms.
        end_timestamp (int): End of time range in Unix ms.
        page (int): Page number for paginated response.

    Returns:
        dict: API response as a dictionary, or raises exception on failure.
    """
    try:
        base_url = "https://posistapi.com/api/v1/pos"
        endpoint = "bills"
        auth_type = "bearer_token"

        # Initialise API client with parameters
        client = APIClient(
            base_url=base_url,
            auth={"type": auth_type, "access_token": access_token}
        )

        params = {
            "customer_key": customer_key,
            "from": start_timestamp,
            "to": end_timestamp,
            "page": page
        }

        response = client.get(endpoint, params)
        return response

    except Exception as e:
        logger.error(f"Error fetching data from {endpoint}: {e}")
        raise

In [ ]:
# Convert a Row object to a Python dictionary
def row_to_dict(row):
    """
    Recursively converts a PySpark Row object to a Python dictionary.

    Args:
        row: A PySpark Row, list, or primitive value.

    Returns:
        dict or list or primitive: Converted Python object.
    """
    if isinstance(row, Row):
        return {k: row_to_dict(v) for k, v in row.asDict().items()}
    elif isinstance(row, list):
        return [row_to_dict(item) for item in row]
    else:
        return row

In [ ]:
# Update watermark_old and watermark_new in config file
def update_config_file(config, watermark_old, history_load_enabled):
    """
    Updates the watermark_old value and disables history_load in the config file on ADLS.
    Called after a successful ingestion run to advance the watermark.

    Args:
        config (Row): Current config Row object read from ADLS.
        watermark_old (str): New watermark_old value to persist (previous run's end timestamp as date string).
        history_load_enabled (bool): Whether history load was active for this run.
    """
    config_dict = row_to_dict(config)

    if history_load_enabled == True:
        config_dict["history_load"]["start_date"] = watermark_old
        if config_dict["history_load"]["start_date"] == config_dict["history_load"]["end_date"]:
            config_dict["history_load"]["enabled"] = False
    else:
        config_dict["watermark_old"] = watermark_old

    dbutils.fs.put(
        f"abfss://{container}@{storage_account}.dfs.core.windows.net/{config_relative_path}/posist_{outlet}_config.json",
        json.dumps(config_dict, indent=4),
        True
    )

## Fetch parameter values

Parameters are passed via Databricks widgets, making this notebook runnable from a pipeline or job with different outlet configs.

In [ ]:
# Set parameters
dbutils.widgets.text("storage_account", "storage_account")
dbutils.widgets.text("container", "container")
dbutils.widgets.text("outlet", "outlet")
dbutils.widgets.text("config_path", "config_path")

# Fetch parameter values
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")
outlet = dbutils.widgets.get("outlet")
config_relative_path = dbutils.widgets.get("config_path")

logger.info(f"Storage account: {storage_account}")
logger.info(f"Container: {container}")
logger.info(f"Outlet: {outlet}")

## Fetch values from config file

Reads the outlet-specific config JSON from ADLS. All ingestion parameters — access token, customer key, watermarks, target path, history load settings — are driven from this file.

In [ ]:
# Define config file path
config = spark.read.option("multiline", "true").json(
    f"abfss://{container}@{storage_account}.dfs.core.windows.net/{config_relative_path}/posist_{outlet}_config.json"
).collect()[0]

# Fetch values from config
customer_key       = config["customer_key"]
access_token       = config["access_token"]
target_relative_path = config["target_path"]
watermark_old      = config["watermark_old"]
watermark_new      = config["watermark_new"]

history_load_enabled    = config["history_load"]["enabled"]
history_load_start_date = config["history_load"]["start_date"]
history_load_end_date   = config["history_load"]["end_date"]

# Define target path on ADLS
target_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{target_relative_path}"

## Main execution block

**Incremental mode** (`history_load.enabled = false`): fetches data between `watermark_old` and `watermark_new`.

**History load mode** (`history_load.enabled = true`): fetches data between `history_load.start_date` and `history_load.end_date`.

For each day in the range, the notebook pages through the API response until no more data is returned, writing each page as a separate JSON file to the ADLS landing layer.

In [ ]:
# Main execution block
if __name__ == "__main__":
    try:
        logger.info("Starting main execution")

        # Get batch id
        batch_id = dbutils.jobs.taskValues.get(taskKey="initialise_ingestion_process", key="batch_id")
        batch_id = f"{get_current_date()[:10].replace('-', '')}_{outlet}_bills"

        # Process date
        process_date = get_current_date()[:10]

        # Get start and end timestamps for the data retrieval
        if history_load_enabled == True:
            unix_timestamps = get_unix_timestamps(history_load_start_date, history_load_end_date)
            logger.info(f"Fetching bills for date {history_load_start_date} - {history_load_end_date}")
        else:
            unix_timestamps = get_unix_timestamps(watermark_old, watermark_new)
            logger.info(f"Fetching bills for date {watermark_old} - {watermark_new}")

        # Process each timestamp range (one per day)
        for timestamps in unix_timestamps:
            start_timestamp = timestamps[0]
            end_timestamp   = timestamps[1]

            bills_date = convert_unix_timestamp_to_date(start_timestamp)[:10]
            response   = "response"
            flag       = 0
            page       = 1

            logger.info(f"Fetching bills for {outlet} for date {bills_date}")

            # Fetch bills in pages until no more response is returned
            while response != []:
                response = get_bills(access_token, customer_key, start_timestamp, end_timestamp, page)

                if response != []:
                    write_api_response(
                        response,
                        f"{target_path}/{process_date}/posist_bills_{outlet}_{bills_date.replace('-', '')}_{page}.json"
                    )
                    flag = 1

                page += 1

            if flag == 1:
                logger.info(f"Bills successfully fetched for {outlet} for date {bills_date}")
            else:
                logger.info(f"No bills found for {outlet} for date {bills_date}")

            # Advance watermark_old to the end of this period
            watermark_old = convert_unix_timestamp_to_date(end_timestamp)

        logger.info(f"Bills successfully fetched for {outlet}")

        # Update config file with new watermark_old after successful run
        update_config_file(config, watermark_old, history_load_enabled)

        logger.info("Main execution completed successfully")

    except Exception as e:
        update_config_file(config, watermark_old, history_load_enabled)
        logger.error(f"Error fetching bills for {outlet}: {e}")
        raise